In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

# Load and inspect

In [3]:
rs_df = pd.read_csv(
    "datasets/review_sentiment.csv"
)
df = rs_df.copy()

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


# Preprocessing

### Missing and duplicate values

No need to care for inconsistent values.

In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(418)

Duplicates should be removed

In [8]:
df = df.drop_duplicates()

### Cleaning

In [9]:
# To lower case
df["review"] = df["review"].str.lower()

In [10]:
# Removing URLs
def remove_url(text):
    return re.sub(r"http\S+", "", text)

df["review"] = df["review"].map(remove_url)

In [11]:
# Removing punctuation
def remove_puntucation(text):
    return re.sub(r"[^A-Za-z0-9\s]", "", text)

df["review"] = df["review"].map(remove_puntucation)

In [12]:
# Remove html
# Removing punctuation
def remove_html(text):
    return re.sub(r"<.*?>", "", text)

df["review"] = df["review"].map(remove_html)

### Tokenize

In [13]:
df["review"] = df["review"].map(word_tokenize)

### Remove stopwords

In [14]:
stop_words = stopwords.words("english")
def remove_stopwords(tokens):
    filtered = []
    for token in tokens:
        if token not in stop_words:
            filtered.append(token)

    return filtered

df["review"] = df["review"].map(remove_stopwords)
        

### Stemming

In [15]:
stemmer = PorterStemmer()
def stem(tokens):
    stemmed = []
    for token in tokens:
        stemmed.append(stemmer.stem(token))

    return stemmed

df["review"] = df["review"].map(stem)

### Vectorization

In [16]:
# vectorizer expects string docs so we need a dummy function
def dummy(doc):
    return doc

vectorizer = TfidfVectorizer(
    max_features=5000,
    tokenizer=dummy,
    preprocessor=dummy,
    lowercase=False,
    token_pattern=None
)

X = vectorizer.fit_transform(df["review"])

### Encoding output

In [17]:
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

### Convert to tensor -> dataset -> dataloader

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
# convert to numpy array
X_train = X_train.toarray()
X_test = X_test.toarray()

In [20]:
# to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [21]:
# to dataset
train_set = TensorDataset(X_train_tensor, y_train_tensor)
test_set = TensorDataset(X_test_tensor, y_test_tensor)

In [22]:
# to dataloaders
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# RNN Model

## Define Model

In [ ]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size=128, num_layers=1, output_size=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size, # number of features in input
            hidden_size, # number of neurons in each hidden layer
            num_layers, # number of hidden layers
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            output_size
        )

    def forward(self, x):
        # It will be initialized only during the first time
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, (hn, cn) = self.lstm(x, (h0, c0)) # out => hidden state of time stamps (batch_size, seq_len, hidden_size)

        return self.fc(out[:, -1, :])




## Training

In [41]:
rnn_model = RNN(
    X_train.shape[1]
)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(rnn_model.parameters())

In [42]:
epochs = 10

for epoch in range(0, epochs):
    rnn_model.train()
    train_loss = 0

    for Xb, yb in train_loader:

        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add extra dimension to make it 3d

        outputs = rnn_model(Xb) # predict logits

        y_pred = torch.sigmoid(outputs.squeeze()) # convert to probabilities 

        loss = criterion(y_pred, yb) # loss calculation

        loss.backward() # back propagation

        optimizer.step() # gradient descent

        train_loss += loss.item()

    print(f"Epoch {epoch} => {train_loss / len(train_loader)}")

    

Epoch 0 => 0.6054548847098504
Epoch 1 => 0.5499925499481539
Epoch 2 => 0.5419379449659778
Epoch 3 => 0.5372316637346821
Epoch 4 => 0.5339642007985422
Epoch 5 => 0.5315872811021344
Epoch 6 => 0.5296459711367084
Epoch 7 => 0.5286042542226853
Epoch 8 => 0.5274429813988747
Epoch 9 => 0.5266763418432205


## Evaluation

In [43]:
rnn_model.eval()

y_actual = []
y_pred = []

with torch.no_grad():

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        y_pred_logits = rnn_model(Xb)
        y_pred_probs = torch.sigmoid(y_pred_logits.squeeze())
        y_pred_labels = (y_pred_probs >= 0.5).float()

        y_actual.extend(yb.cpu().numpy())
        y_pred.extend(y_pred_labels.cpu().numpy())

In [44]:
print(f"Accuracy: {accuracy_score(y_actual, y_pred)}")
print(classification_report(y_actual, y_pred))

Accuracy: 0.8722395885852576
              precision    recall  f1-score   support

         0.0       0.85      0.90      0.88      4939
         1.0       0.89      0.85      0.87      4978

    accuracy                           0.87      9917
   macro avg       0.87      0.87      0.87      9917
weighted avg       0.87      0.87      0.87      9917

